In [1]:
import pandas as pd
import numpy as np
# import gc
# import os

In [ ]:
sales=pd.read_feather(r"src")

In [10]:
sales=sales.sort_values(['item_id','store_id','date']).reset_index(drop=True)
sales

,item_id,store_id,dept_id,cat_id,state_id,sales,date,sell_price,event_name_1,event_type_1,snap_CA
0,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,3,2011-01-29,2.00,No event,NaN,0
1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-01-30,2.00,No event,NaN,0
2,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-01-31,2.00,No event,NaN,0
3,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,1,2011-02-01,2.00,No event,NaN,1
4,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,4,2011-02-02,2.00,No event,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...
59181085,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-18,5.94,No event,NaN,0
59181086,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-19,5.94,No event,NaN,0
59181087,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-20,5.94,No event,NaN,0
59181088,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-21,5.94,No event,NaN,0


In [ ]:
lag_days=[7,14,28]
for lag in lag_days:
    sales[f'lag_{lag}'] = (
        sales.groupby(['item_id','store_id'])['sales']
        .shift(lag)
        .astype('float32')
    )

print(f"Lag features done — nulls introduced: {sales[['lag_7','lag_14','lag_28']].isnull().sum().to_dict()}")

In [12]:
for window in[7,28]:
    sales[f'roll_mean_{window}']=(
    sales.groupby(['item_id','store_id'])['sales']
    .transform(lambda x:x.shift(1).rolling(window,min_periods=1).mean())
    .astype('float32')
    )
    sales[f'roll_std_{window}']=(
        sales.groupby(['item_id','store_id'])['sales']
        .transform(lambda x: x.shift(1).rolling(window,min_periods=(1)).std())
        .astype('float32')
    )
print('Done with rolling features')

C:\Users\nevil\AppData\Local\Temp\ipykernel_7908\1523997514.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sales.groupby(['item_id','store_id'])['sales']
C:\Users\nevil\AppData\Local\Temp\ipykernel_7908\1523997514.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sales.groupby(['item_id','store_id'])['sales']
C:\Users\nevil\AppData\Local\Temp\ipykernel_7908\1523997514.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warni

Done with rolling features


In [13]:
import os
import gc

In [14]:
gc.collect()

0

In [15]:
sales['day_of_week']=sales['date'].dt.dayofweek.astype('int8')
sales['month']=sales['date'].dt.month.astype('int8')
sales['year']=sales['date'].dt.year.astype('int16')
sales['week_of_yr']=sales['date'].dt.isocalendar().week.astype('int8')
sales['is_weekend']=(sales['day_of_week'] >= 5).astype('int8')
sales['quarter']=sales['date'].dt.quarter.astype('int8')
print('calendar features done')

calendar features done


In [16]:
sales['snap_CA'] = sales['snap_CA'].astype('int8')
sales['has_event']=(sales['event_name_1']!='No event').astype('int8')
print(sales['has_event'].value_counts())
event_type_map={
    'Sporting':1,
    'Cultural':2,
    'National':3,
    'Religious':4
}
sales['event_type_enc']=(
    sales['event_type_1']
    .map(event_type_map)
    .astype('float32')
    .fillna(0)
    .astype('int8')
)
print(sales['event_type_enc'].value_counts())



has_event
0    54363670
1     4817420
Name: count, dtype: int64
event_type_enc
0    54363670
4     1646460
3     1554990
2     1128130
1      487840
Name: count, dtype: int64


In [17]:
sales['price_change']=(
    sales.groupby(['item_id','store_id'])['sell_price']
    .transform(lambda x:x.pct_change())
    .astype('float32')
)
sales['price_rel_mean']=(
    sales['sell_price']/
    sales.groupby('item_id')['sell_price'].transform('mean')
    .astype('float32')
)
sales['price_direction'] = (
    sales.groupby(['item_id','store_id'])['sell_price']
    .transform(lambda x: x.diff().apply(lambda v: 1 if v > 0 else (-1 if v < 0 else 0)))
    .astype('int8')
)


C:\Users\nevil\AppData\Local\Temp\ipykernel_7908\4103795176.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sales.groupby(['item_id','store_id'])['sell_price']
C:\Users\nevil\AppData\Local\Temp\ipykernel_7908\4103795176.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sales.groupby('item_id')['sell_price'].transform('mean')
C:\Users\nevil\AppData\Local\Temp\ipykernel_7908\4103795176.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and si

In [18]:
print(f"Memory   : {sales.memory_usage(deep=True).sum()/1e9:.2f} GB")

Memory   : 3.73 GB


In [19]:
print(f"Shape:{sales.shape}")
print(f"Memory : {sales.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Nulls  :\n{sales.isnull().sum()[sales.isnull().sum() > 0]}")


Shape:(59181090, 29)
Memory : 3.73 GB
Nulls  :
event_type_1    54363670
lag_7             213430
lag_14            426860
lag_28            853720
roll_mean_7        30490
roll_std_7         60980
roll_mean_28       30490
roll_std_28        60980
price_change       30490
dtype: int64


In [20]:
event_categories=[c for c in sales['event_name_1'].cat.categories if c!= 'No event']
event_map={'No event':0}
event_map.update({event:i+1 for i,event in enumerate(event_categories)})
sales['event_name_enc']=sales['event_name_1'].map(event_map).astype('int8')



In [21]:

print(sales['event_name_enc'].value_counts().head(15))

event_name_enc
0     54363670
29      182940
23      182940
12      182940
13      182940
26      182940
24      182940
22      182940
16      182940
27      182940
2       152450
7       152450
30      152450
28      152450
20      152450
Name: count, dtype: int64


In [22]:
print(event_map)

{'No event': 0, 'Chanukah End': 1, 'Christmas': 2, 'Cinco De Mayo': 3, 'ColumbusDay': 4, 'Easter': 5, 'Eid al-Fitr': 6, 'EidAlAdha': 7, "Father's day": 8, 'Halloween': 9, 'IndependenceDay': 10, 'LaborDay': 11, 'LentStart': 12, 'LentWeek2': 13, 'MartinLutherKingDay': 14, 'MemorialDay': 15, "Mother's day": 16, 'NBAFinalsEnd': 17, 'NBAFinalsStart': 18, 'NewYear': 19, 'OrthodoxChristmas': 20, 'OrthodoxEaster': 21, 'Pesach End': 22, 'PresidentsDay': 23, 'Purim End': 24, 'Ramadan starts': 25, 'StPatricksDay': 26, 'SuperBowl': 27, 'Thanksgiving': 28, 'ValentinesDay': 29, 'VeteransDay': 30}


In [23]:
sales=sales.drop(columns=['event_name_1','event_type_1'])
sales

,item_id,store_id,dept_id,cat_id,state_id,sales,date,sell_price,snap_CA,lag_7,...,year,week_of_yr,is_weekend,quarter,has_event,event_type_enc,price_change,price_rel_mean,price_direction,event_name_enc
0,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,3,2011-01-29,2.00,0,NaN,...,2011,4,1,1,0,0,NaN,0.923653,0,0
1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-01-30,2.00,0,NaN,...,2011,4,1,1,0,0,0.0,0.923653,0,0
2,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-01-31,2.00,0,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
3,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,1,2011-02-01,2.00,1,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
4,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,4,2011-02-02,2.00,1,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59181085,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-18,5.94,0,0.0,...,2016,20,0,2,0,0,0.0,1.000601,0,0
59181086,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-19,5.94,0,0.0,...,2016,20,0,2,0,0,0.0,1.000601,0,0
59181087,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-20,5.94,0,0.0,...,2016,20,0,2,0,0,0.0,1.000601,0,0
59181088,HOUSEHOLD_2_516,WI_3,HOUSEHOLD_2,HOUSEHOLD,WI,0,2016-05-21,5.94,0,1.0,...,2016,20,1,2,0,0,0.0,1.000601,0,0


In [24]:
sales.head(10)

,item_id,store_id,dept_id,cat_id,state_id,sales,date,sell_price,snap_CA,lag_7,...,year,week_of_yr,is_weekend,quarter,has_event,event_type_enc,price_change,price_rel_mean,price_direction,event_name_enc
0,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,3,2011-01-29,2.0,0,NaN,...,2011,4,1,1,0,0,NaN,0.923653,0,0
1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-01-30,2.0,0,NaN,...,2011,4,1,1,0,0,0.0,0.923653,0,0
2,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-01-31,2.0,0,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
3,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,1,2011-02-01,2.0,1,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
4,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,4,2011-02-02,2.0,1,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
5,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,2,2011-02-03,2.0,1,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
6,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-02-04,2.0,1,NaN,...,2011,5,0,1,0,0,0.0,0.923653,0,0
7,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,2,2011-02-05,2.0,1,3.0,...,2011,5,1,1,0,0,0.0,0.923653,0,0
8,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-02-06,2.0,1,0.0,...,2011,5,1,1,1,1,0.0,0.923653,0,27
9,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,0,2011-02-07,2.0,1,0.0,...,2011,6,0,1,0,0,0.0,0.923653,0,0


In [25]:
import gc
gc.collect()


0